# Train BPR Model

Train and save the BPR checkpoint using local DWH mock data.

In [ ]:
import os
import sys
from pathlib import Path

import mlflow
import mlflow.pyfunc
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/Workspace/Users/vminh9560@gmail.com/recommender-service")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data_test" / "dwh_mock"
BPR_PATH = PROJECT_ROOT / "models" / "bpr" / "model.npz"
os.environ["BPR_CHECKPOINT_PATH"] = str(BPR_PATH)

from src.db import LocalDataClient
from src.pipeline import build_interaction_table, train_bpr, save_bpr_model
from src.fallback import filter_cold_books

class BPRPyfunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        data = np.load(context.artifacts["checkpoint"], allow_pickle=False)
        self.user_factors = data["user_factors"]
        self.item_factors = data["item_factors"]
        self.user_ids = data["user_ids"].astype(int).tolist()
        self.item_ids = data["item_ids"].astype(int).tolist()
        self.user_index = {uid: idx for idx, uid in enumerate(self.user_ids)}

    def predict(self, context, model_input):
        df = pd.DataFrame(model_input)
        if "user_id" not in df.columns:
            raise ValueError("model_input must include user_id")
        top_k_series = df.get("top_k")
        results = []
        for i, row in df.iterrows():
            user_id = int(row["user_id"])
            top_k = int(top_k_series.iloc[i]) if top_k_series is not None else 5
            u_idx = self.user_index.get(user_id)
            if u_idx is None:
                results.append([])
                continue
            scores = self.item_factors.dot(self.user_factors[u_idx])
            top_idx = np.argsort(scores)[::-1][:top_k]
            recs = [
                {
                    "book_id": int(self.item_ids[j]),
                    "mf_score": float(scores[j]),
                    "rank": rank + 1,
                }
                for rank, j in enumerate(top_idx)
            ]
            results.append(recs)
        return pd.DataFrame({"recommendations": results})

with LocalDataClient(DATA_DIR) as client:
    interaction_df = build_interaction_table(client)
    interaction_df, _ = filter_cold_books(interaction_df)
    bpr_model = train_bpr(interaction_df)
    save_bpr_model(bpr_model, str(BPR_PATH))

mlflow.set_registry_uri("databricks-uc")
registered_model_name = "main.default.bpr_retrieval"
with mlflow.start_run(run_name="bpr-train") as run:
    mlflow.log_artifact(str(BPR_PATH), artifact_path="checkpoint")
    mlflow.pyfunc.log_model(
        artifact_path="bpr_model",
        python_model=BPRPyfunc(),
        artifacts={"checkpoint": str(BPR_PATH)},
        registered_model_name=registered_model_name,
        pip_requirements=["mlflow", "numpy", "pandas"],
    )
    print("MLflow run:", run.info.run_id)

print("Saved BPR checkpoint to:", BPR_PATH)
print("Registered model:", registered_model_name)

In [ ]:
# Register BGE reranker from local checkpoint using MLflow
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

BGE_PATH = PROJECT_ROOT / "models" / "bge-reranker"
registered_reranker_name = "main.default.bge_base_reranker"

class BGERerankerPyfunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        model_path = context.artifacts["checkpoint"]
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.model.eval()
        self.device = torch.device("cpu")
        self.model.to(self.device)

    def predict(self, context, model_input):
        df = pd.DataFrame(model_input)
        if "user_profile" not in df.columns or "item_description" not in df.columns:
            raise ValueError("model_input must include user_profile and item_description")
        encoded = self.tokenizer(
            df["user_profile"].tolist(),
            df["item_description"].tolist(),
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        encoded = {k: v.to(self.device) for k, v in encoded.items()}
        with torch.no_grad():
            logits = self.model(**encoded).logits.squeeze(-1)
            scores = torch.sigmoid(logits).cpu().numpy().tolist()
        return pd.DataFrame({"score": scores})

mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run(run_name="bge-reranker-register") as run:
    mlflow.pyfunc.log_model(
        artifact_path="bge_reranker",
        python_model=BGERerankerPyfunc(),
        artifacts={"checkpoint": str(BGE_PATH)},
        registered_model_name=registered_reranker_name,
        pip_requirements=["mlflow", "numpy", "pandas", "torch", "transformers"],
    )
    print("MLflow run:", run.info.run_id)

print("Registered reranker:", registered_reranker_name)